In [1]:
!pip install -U langchain-google-genai
%pip install dotenv


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from pathlib import Path
import os

# Load environment variables
env_path = Path("../.env").resolve()
load_dotenv(env_path, override=True)

# Check API key
print("Environment file exists:", env_path.exists())
print("Google API key loaded:", bool(os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")))

# Initialize LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)

# Initialize test case LLM
tc_llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)

Environment file exists: True
Google API key loaded: True


In [3]:
from langchain.tools import tool
from langchain_core.prompts import PromptTemplate
import os
from langchain_community.document_loaders import (
    UnstructuredPDFLoader
)


@tool
def generate_test_cases_from_pdf() -> str:
    "Reads the requirements from the PDF and generate atleast 10 test scenarios in BDD Format"

    docs_folder = "./Docs"
    documents = []

    for filename in os.listdir(docs_folder):
        filepath = os.path.join(docs_folder, filename)
        loader = UnstructuredPDFLoader(filepath)
        docs = loader.load();
        for doc in docs:
            documents.append(f"{filename}:\n{doc.page_content.strip()}")

    requirement_txt = "\n\n".join(documents[:3])[:1000]


    prompt_template = PromptTemplate.from_template(

        """
        You are are QA Automation Engineer.
        Your task is to convert the following user stories into at least 10 testcases, in Gherkin BDD style format.
        Include combinations of valid, invalid, edge case and alternative flow scearios

        {requirement_txt}

        Format:
        Feature: ...
        Scenario: ...
            Given ...
            When ...
            Then ...

        """
    )
    prompt = prompt_template.invoke({"requirement_txt": requirement_txt})
    return tc_llm.invoke(prompt).content


/var/folders/br/9g69f18n36b4rhnfj1_s7fnc0000gn/T/ipykernel_36630/1848013738.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [4]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Modern agent (replaces deprecated initialize_agent + AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)
agent = create_agent(
    tools=[generate_test_cases_from_pdf],
    model=llm,
)

result = agent.invoke(
    {"messages": [HumanMessage(content="Generate the test cases from PDF file by reading the requirement pdf documents")]}
)

print(result["messages"][-1].content)


Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1
No languages specified, defaulting to English.
No languages specified, defaulting to English.
No languages specified, defaulting to English.


[{'type': 'text', 'text': 'The requirement specifications from the PDF document have been analyzed, and 11 comprehensive BDD test scenarios have been generated across key architectural functional areas:\n\n---\n\n### **Feature: Netflix Video Ingestion, Transcoding, and Streaming Pipeline**\n\n```gherkin\nFeature: Netflix Video Ingestion, Transcoding, and Streaming Pipeline\n  As a streaming platform subscriber and operations team\n  We want a resilient, distributed media pipeline and content delivery system\n  So that high-quality, uninterrupted playback is delivered across heterogeneous devices and network conditions.\n\n  # -------------------------------------------------------------\n  # Feature Area 1: Video Ingestion & Transcoding Process\n  # -------------------------------------------------------------\n\n  @valid @ingestion @transcoding\n  Scenario: Successfully transcode a mezzanine video file into multi-bitrate chunks\n    Given a valid master mezzanine video file "movie_mas